In [ ]:
import gc
import sklearn
import imblearn
import numpy as np
import keras_tuner
import tensorflow as tf
from collections import Counter
import matplotlib.pyplot as plt
import Modules.constants as constants
import Modules.ds_loader as ds_loader
print(os.cpu_count())
SAMPLE_PERCENTAGE = 1.0

DATA_PATH = constants.DATASET

TRAIN_DIR = DATA_PATH / "train"
VAL_DIR = DATA_PATH / "val"
TEST_DIR = DATA_PATH / "test"

X_train, y_train = ds_loader.load_data(TRAIN_DIR)
X_test, y_test = ds_loader.load_data(TEST_DIR)
X_val, y_val = ds_loader.load_data(VAL_DIR)

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

2025-04-14 09:11:52.285949: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-14 09:11:52.323762: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744614712.344642  237522 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744614712.351487  237522 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744614712.369022  237522 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

✅ Loaded 7788 samples with shape (500, 12)
✅ Loaded 1850 samples with shape (500, 12)
✅ Loaded 1855 samples with shape (500, 12)


In [2]:
def remove_flatlined_sequences(X_train, y_train, X_val, y_val, X_test, y_test,threshold):
    def is_flatlined(sequence, threshold):
        if np.max(sequence) - np.min(sequence) < threshold:
            return True
        
        if np.all(sequence == sequence[0]):
            return True
        
        return False
    
    original_train_size = len(X_train)
    original_val_size = len(X_val)
    original_test_size = len(X_test)

    train_mask = [not is_flatlined(seq, threshold) for seq in X_train]
    val_mask = [not is_flatlined(seq, threshold) for seq in X_val]
    test_mask = [not is_flatlined(seq, threshold) for seq in X_test]

    X_train = np.array([seq for seq, keep in zip(X_train, train_mask) if keep])
    y_train = np.array([label for label, keep in zip(y_train, train_mask) if keep])
    
    X_val = np.array([seq for seq, keep in zip(X_val, val_mask) if keep])
    y_val = np.array([label for label, keep in zip(y_val, val_mask) if keep])
    
    X_test = np.array([seq for seq, keep in zip(X_test, test_mask) if keep])
    y_test = np.array([label for label, keep in zip(y_test, test_mask) if keep])

    removed_train = original_train_size - len(X_train)
    removed_val = original_val_size - len(X_val)
    removed_test = original_test_size - len(X_test)
    
    print(f"Original number of sequences in X_train: {original_train_size}")
    print(f"Removed flatlined sequences from X_train: {removed_train}")
    print(f"Original number of sequences in X_val: {original_val_size}")
    print(f"Removed flatlined sequences from X_val: {removed_val}")
    print(f"Original number of sequences in X_test: {original_test_size}")
    print(f"Removed flatlined sequences from X_test: {removed_test}")

    return X_train, y_train, X_val, y_val, X_test, y_test


threshold = 0.2 
X_train, y_train, X_val, y_val, X_test, y_test = remove_flatlined_sequences(X_train, y_train, X_val, y_val, X_test, y_test, threshold)


Original number of sequences in X_train: 7788
Removed flatlined sequences from X_train: 0
Original number of sequences in X_val: 1855
Removed flatlined sequences from X_val: 0
Original number of sequences in X_test: 1850
Removed flatlined sequences from X_test: 0


In [3]:
print("Unique classes in y:", np.unique(y_train))
print("Datatype:", print(X_train.dtype), print(y_train.dtype))
print(f"Min and Max of X_train: {np.min(X_train)}, {np.max(X_train)}")
print(f"Min and Max of X_val: {np.min(X_val)}, {np.max(X_val)}")
print(f"Min and Max of X_test: {np.min(X_test)}, {np.max(X_test)}")
print(f"NaNs in X: {np.isnan(X_train).sum()}")
print(f"Infs in X: {np.isinf(X_train).sum()}")
print(f"Class distribution: {Counter(y_train)}")

Unique classes in y: [0 1 2 3]
float32
int32
Datatype: None None
Min and Max of X_train: 0.0, 1.0000001192092896
Min and Max of X_val: -8.730203628540039, 11.054253578186035
Min and Max of X_test: -11.602144241333008, 13.575825691223145
NaNs in X: 0
Infs in X: 0
Class distribution: Counter({np.int32(2): 2811, np.int32(1): 1801, np.int32(3): 1594, np.int32(0): 1582})


In [ ]:
X_train_flat = X_train.reshape((X_train.shape[0],-1))
smote = imblearn.over_sampling.SMOTE(random_state=42)
X_resampled, y_train = smote.fit_resample(X_train_flat, y_train)
X_train = X_resampled.reshape((-1, *X_train.shape[1:]))

print(f"Class distribution: {Counter(y_train)}")

In [5]:
# 1-D convolutional ResNet model 
# https://pmc.ncbi.nlm.nih.gov/articles/PMC10128986/#sec012
class Resnet(keras_tuner.HyperModel):
    def residual_block(self, inputs, c_units, p_units, k_units):
        # C1 BLOCK
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(inputs)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        
        x = tf.keras.layers.Conv1D(filters=c_units, kernel_size=k_units, strides=1, padding='same')(x)
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        # SC
        s = tf.keras.layers.Conv1D(filters=c_units, kernel_size=1, strides=1, padding='same')(inputs)
        x = tf.keras.layers.Add()([x, s])
        x = tf.keras.layers.ReLU()(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling1D(p_units, strides=2)(x)
        return x


    def build(self, hp):
        gc.collect()
        tf.keras.backend.clear_session()
        # HYPERPARAMS
        n_layer = 3
        k_units = 3
        p_units = 5
        #p_units = hp.Int("p_units", min_value=2, max_value=10, step=1)
        c_units = hp.Choice("c_units", [128])
        d_units_0 = hp.Choice("d_units_0", [512])
        d_units_1 = hp.Choice('d_units_coef', [2,4])
        dropout_0 = hp.Float('dropout_0', min_value = 0.3, max_value=0.5, step=0.05)
        dropout_1 = hp.Float('dropout_1', min_value = 0.3, max_value=0.5, step=0.05)
        
        # INPUT LAYER
        inputs = tf.keras.Input(shape=(500,12))
        
        # RESIDUALS
        x = self.residual_block(inputs, c_units, p_units, k_units)
        filter_size = c_units
        for i in range(1, n_layer):
            #filter_size *= 2  
            x = self.residual_block(x, filter_size, p_units, k_units)

        # CLASSIFIER
        x = tf.keras.layers.Flatten()(x)
        x = tf.keras.layers.Dense(d_units_0, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_0)(x)  
        x = tf.keras.layers.Dense(d_units_0 // d_units_1, activation='relu')(x)
        x = tf.keras.layers.Dropout(dropout_1)(x)  

        # OUTPUT
        outputs = tf.keras.layers.Dense(4, activation='softmax')(x)
        
        model = tf.keras.Model(inputs, outputs)
        optimizer = tf.keras.optimizers.Adam(
                        learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-3),
                        weight_decay=hp.Choice('weight_decay',[1e-3,1e-4,1e-5,0.0])
                    )
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        
        
        return model
    
    def fit(self, hp, model, *args, **kwargs):
        return model.fit(
            batch_size= hp.Choice("batch_size", [32]),
            *args,
            **kwargs,
        ) 

In [6]:
RDIR="Results/RES_W5000_L3/" 
MDIR= RDIR + "RES_W5000_L3.keras"
CDIR= RDIR + "C_RES_W5000_L3.keras"
CVDIR = RDIR + "RES_W5000_L3_CV.keras"
tuner = keras_tuner.BayesianOptimization(
    Resnet(),
    max_trials=20,
    overwrite=False,
    objective='val_accuracy',
    directory=RDIR,
    project_name="RES_W5000_L3_00",
    )

tuner.search_space_summary()

Reloading Tuner from Results/RES_W5000_L3/RES_W5000_L3_00/tuner0.json
Search space summary
Default search space size: 8
c_units (Choice)
{'default': 64, 'conditions': [], 'values': [64], 'ordered': True}
d_units_0 (Choice)
{'default': 128, 'conditions': [], 'values': [128], 'ordered': True}
d_units_coef (Choice)
{'default': 2, 'conditions': [], 'values': [2, 4], 'ordered': True}
dropout_0 (Float)
{'default': 0.3, 'conditions': [], 'min_value': 0.3, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
dropout_1 (Float)
{'default': 0.3, 'conditions': [], 'min_value': 0.3, 'max_value': 0.5, 'step': 0.05, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.001, 'step': None, 'sampling': 'linear'}
weight_decay (Choice)
{'default': 0.001, 'conditions': [], 'values': [0.001, 0.0001, 1e-05, 0.0], 'ordered': True}
batch_size (Choice)
{'default': 32, 'conditions': [], 'values': [32], 'ordered': True}


I0000 00:00:1744614767.462310  237522 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2381 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [7]:
callback_list = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy",mode="max", restore_best_weights=True,patience=5, verbose=0),
    tf.keras.callbacks.ModelCheckpoint(filepath=CDIR,monitor='val_accuracy', save_best_only=True, save_weights_only=False,    
    verbose=0)
]
tuner.search(
    X_train, y_train, 
    epochs = 150,
    validation_data=(X_val, y_val),
    callbacks=callback_list 
)


Search: Running Trial #9

Value             |Best Value So Far |Hyperparameter
64                |64                |c_units
128               |128               |d_units_0
2                 |2                 |d_units_coef
0.5               |0.45              |dropout_0
0.4               |0.5               |dropout_1
0.00051849        |0.00030197        |learning_rate
0                 |0                 |weight_decay
32                |32                |batch_size



Traceback (most recent call last):
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copied_kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/tuner.py", line 233, in _build_and_fit_model
    results = self.hypermodel.fit(hp, model, *args, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/tuner.py", line 314, in run_trial
    obj_value = self._build_and_fit_model(trial, *args, **copied_kwargs)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/capitan/.venv/tenv/lib64/python3.11/site-packages/keras_tuner/src/engine/tuner.py", line 233, in _build_and_fit_model
    results = self.hypermodel.f

In [ ]:
tuner.results_summary()

In [ ]:
models = tuner.get_best_models(num_models=1)
best_model = models[0]
best_model.summary()
best_model.save(MDIR) 

In [ ]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 
print(best_hps.values)

In [ ]:
test_loss, test_accuracy = best_model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = best_model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = best_model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns
y_pred_class = np.argmax(y_pred, axis=1)  
cm = sklearn.metrics.confusion_matrix(y_test, y_pred_class, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
kfold = sklearn.model_selection.StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
fold_accuracies = []
fold_histories = []

best_accuracy = 0.0
best_model = None  

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"\n--- Fold {fold+1} ---")

    fold_callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    ]
    X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
    y_tr, y_val_fold = y_train[train_idx], y_train[val_idx]

    model = Resnet().build(best_hps)

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_val_fold, y_val_fold),
        epochs=100,
        callbacks=fold_callbacks,
        verbose=1
    )

    val_loss, val_accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    print(f"Fold {fold+1} Validation Accuracy: {val_accuracy:.4f}")
    fold_accuracies.append(val_accuracy)
    fold_histories.append(history)

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = model
        model.save(CVDIR) 
        print(f"Saved best model from Fold {fold+1} with Accuracy: {val_accuracy:.4f}")


In [ ]:
print("Cross-validation accuracies:", fold_accuracies)
print("Average CV accuracy:", np.mean(fold_accuracies))
print("Max CV accuracy:", np.max(fold_accuracies))

In [ ]:
callback_list = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy",mode="max", restore_best_weights=True,patience=5, verbose=0)
]
cv_model = tf.keras.models.load_model(CVDIR)
cv_model.fit(X_train, y_train, validation_data=(X_val,y_val), epochs=100, callbacks=callback_list)
cv_model.save(CVDIR)

In [ ]:
cv_model = tf.keras.models.load_model(CVDIR)
cv_model.evaluate(X_test, y_test)

In [ ]:
test_loss, test_accuracy = cv_model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

In [ ]:
y_pred = cv_model.predict(X_test)

if y_pred.shape[1] == 1:  
    y_pred_binary = (y_pred > 0.5).astype(int)
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred)  
else:
    y_pred_binary = np.argmax(y_pred, axis=1)  
    auc = sklearn.metrics.roc_auc_score(y_test, y_pred, multi_class='ovr')

print("Classification Report (Test Data):")
print(sklearn.metrics.classification_report(y_test, y_pred_binary))
print(f"AUC: {auc}")

y_train_pred = cv_model.predict(X_train)
y_train_pred = np.argmax(y_train_pred, axis=1)

print("Classification Report (Train Data):")
print(sklearn.metrics.classification_report(y_train, y_train_pred))

In [ ]:
import seaborn as sns
y_pred_class = np.argmax(y_pred, axis=1)  
cm = sklearn.metrics.confusion_matrix(y_test, y_pred_class, normalize='true')

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()

In [ ]:

fig, ax = plt.subplots(1, 2, figsize=(14, 6))

ax[0].plot(history.history['accuracy'], label='accuracy')
ax[0].plot(history.history['val_accuracy'], label='val_accuracy')
ax[0].set_title('Accuracy vs Val Accuracy')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].legend()

ax[1].plot(history.history['loss'], label='loss')
ax[1].plot(history.history['val_loss'], label='val_loss')
ax[1].set_title('Loss vs Val Loss')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Loss')
ax[1].legend()

plt.tight_layout()
plt.show()